In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LABEL_FILE = (
    PROJECT_ROOT
    / "data"
    / "cxr"
    / "cxr_labeled.csv"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "models"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", PROJECT_ROOT)
print("CXR:", CXR_ROOT)
print("Labels:", LABEL_FILE)

In [ ]:
df = pd.read_csv(LABEL_FILE)

FINDINGS = [
    "pneumonia",
    "pleural_effusion",
    "pneumothorax",
    "consolidation",
    "pulmonary_edema",
]

LABEL_COLUMNS = [
    f"label_{finding}"
    for finding in FINDINGS
]

# Official Train and Test split
original_train_df = df[
    df["split"] == "Train"
].copy()

test_df = df[
    df["split"] == "Test"
].copy()

# Create validation ONLY from the official training set
train_df, val_df = train_test_split(
    original_train_df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

In [ ]:
def get_image_path(relative_path):
    return CXR_ROOT / Path(relative_path)


train_df["image_path"] = (
    train_df["relative_image_path"]
    .apply(get_image_path)
)

test_df["image_path"] = (
    test_df["relative_image_path"]
    .apply(get_image_path)
)

print(train_df["image_path"].iloc[0])
print(train_df["image_path"].iloc[0].exists())

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def add_image_paths(df):
    df = df.copy()

    df["image_path"] = df[
        "relative_image_path"
    ].apply(
        lambda p: CXR_ROOT / Path(p)
    )

    return df


train_df = add_image_paths(train_df)
val_df = add_image_paths(val_df)
test_df = add_image_paths(test_df)


def prepare_dataframe(df):

    paths = df[
        "image_path"
    ].astype(str).values

    labels = df[
        LABEL_COLUMNS
    ].values.astype(np.float32)

    # 0 and 1 = valid labels
    # -1 and NaN = ignored
    masks = np.isin(
        labels,
        [0, 1]
    ).astype(np.float32)

    # Replace ignored labels with 0
    labels = np.where(
        masks == 1,
        labels,
        0.0
    ).astype(np.float32)

    return paths, labels, masks


train_paths, train_labels, train_masks = (
    prepare_dataframe(train_df)
)

val_paths, val_labels, val_masks = (
    prepare_dataframe(val_df)
)

test_paths, test_labels, test_masks = (
    prepare_dataframe(test_df)
)


print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

In [ ]:
train_ids = set(train_df["image_id"])
val_ids = set(val_df["image_id"])
test_ids = set(test_df["image_id"])

print("Train ∩ Validation:",
      len(train_ids & val_ids))

print("Train ∩ Test:",
      len(train_ids & test_ids))

print("Validation ∩ Test:",
      len(val_ids & test_ids))

In [ ]:
def load_image(path, label, mask):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)

    return image, (label, mask)

In [ ]:
def augment_image(image):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.05)
    image = tf.image.random_contrast(
        image,
        lower=0.9,
        upper=1.1
    )
    return image

In [ ]:
def create_dataset(paths, labels, masks, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels, masks)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(paths),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    if training:
        dataset = dataset.map(
            lambda image, target: (
                augment_image(image),
                target
            ),
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

In [ ]:
train_ds = create_dataset(
    train_paths,
    train_labels,
    train_masks,
    training=True
)

val_ds = create_dataset(
    val_paths,
    val_labels,
    val_masks,
    training=False
)

test_ds = create_dataset(
    test_paths,
    test_labels,
    test_masks,
    training=False
)

In [ ]:
print("Unique training labels:", np.unique(train_labels))
print("Unique training masks:", np.unique(train_masks))

print("\nLabel counts:")
print(np.unique(train_labels, return_counts=True))

In [ ]:
images, (labels, masks) = next(iter(train_ds))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Masks:", masks.shape)

In [ ]:
POS_WEIGHT = []

for finding in FINDINGS:

    column = f"label_{finding}"

    train_labels = train_df[column].values

    valid_labels = train_labels[
        np.isin(train_labels, [0, 1])
    ]

    positive = np.sum(valid_labels == 1)
    negative = np.sum(valid_labels == 0)

    weight = negative / max(positive, 1)

    POS_WEIGHT.append(weight)

    print(
        f"{finding:20s} "
        f"positive={positive:4d} "
        f"negative={negative:4d} "
        f"pos_weight={weight:.2f}"
    )

POS_WEIGHT = np.maximum(
    np.array(POS_WEIGHT, dtype=np.float32),
    1.0
)

print("\nFinal POS_WEIGHT:")
print(POS_WEIGHT)

In [ ]:
import tensorflow as tf


def masked_weighted_bce(y_true, y_pred, mask):
    """
    Masked weighted binary cross-entropy.

    y_true : [batch, 8]
    y_pred : [batch, 8]
    mask   : [batch, 8]

    mask = 1 -> label contributes to loss
    mask = 0 -> label is ignored
    """

    epsilon = tf.keras.backend.epsilon()

    y_pred = tf.clip_by_value(
        y_pred,
        epsilon,
        1.0 - epsilon
    )

    # Standard BCE
    bce = -(
        y_true * tf.math.log(y_pred)
        +
        (1.0 - y_true) * tf.math.log(1.0 - y_pred)
    )

    # Positive-class weighting
    weights = (
        y_true * POS_WEIGHT
        +
        (1.0 - y_true)
    )

    weighted_bce = bce * weights

    # Ignore uncertain / missing labels
    masked_loss = weighted_bce * mask

    # Average only over valid labels
    loss = tf.reduce_sum(masked_loss) / (
        tf.reduce_sum(mask) + epsilon
    )

    return loss

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0


NUM_CLASSES = len(FINDINGS)

# EfficientNetB0 backbone
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze pretrained backbone initially
base_model.trainable = False


# Model input
inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3),
    name="image"
)

# EfficientNet
x = base_model(
    inputs,
    training=False
)

# Global pooling
x = layers.GlobalAveragePooling2D()(x)

# Regularization
x = layers.Dropout(0.30)(x)

# Dense feature layer
x = layers.Dense(
    128,
    activation="relu"
)(x)

x = layers.Dropout(0.30)(x)

# 8 independent finding probabilities
outputs = layers.Dense(
    NUM_CLASSES,
    activation="sigmoid",
    name="findings"
)(x)

model = Model(
    inputs=inputs,
    outputs=outputs,
    name="CXR_Finding_Classifier"
)

model.summary()

In [ ]:
dummy_input = tf.random.normal(
    (2, IMG_SIZE, IMG_SIZE, 3)
)

dummy_output = model(dummy_input)

print("Output shape:", dummy_output.shape)
print("Output:")
print(dummy_output.numpy())

In [ ]:
import tensorflow as tf


class MaskedCXRModel(tf.keras.Model):

    def __init__(self, backbone_model):
        super().__init__()
        self.backbone_model = backbone_model

        self.loss_tracker = tf.keras.metrics.Mean(
            name="loss"
        )

    @property
    def metrics(self):
        return [self.loss_tracker]

    def call(self, inputs, training=False):
        return self.backbone_model(
            inputs,
            training=training
        )

    def train_step(self, data):

        images, (y_true, mask) = data

        with tf.GradientTape() as tape:

            y_pred = self(
                images,
                training=True
            )

            loss = masked_weighted_bce(
                y_true,
                y_pred,
                mask
            )

        gradients = tape.gradient(
            loss,
            self.trainable_variables
        )

        self.optimizer.apply_gradients(
            zip(
                gradients,
                self.trainable_variables
            )
        )

        self.loss_tracker.update_state(loss)

        return {
            "loss": self.loss_tracker.result()
        }

    def test_step(self, data):

        images, (y_true, mask) = data

        y_pred = self(
            images,
            training=False
        )

        loss = masked_weighted_bce(
            y_true,
            y_pred,
            mask
        )

        self.loss_tracker.update_state(loss)

        return {
            "loss": self.loss_tracker.result()
        }

In [ ]:
cxr_model = MaskedCXRModel(model)

cxr_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    )
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
history = cxr_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np


# Get predictions on validation set
val_predictions = cxr_model.predict(val_ds)

print("Prediction shape:", val_predictions.shape)

In [ ]:
# Get validation labels and masks directly
val_true = val_labels
val_mask = val_masks


print("\nValidation ROC-AUC:\n")

for i, finding in enumerate(FINDINGS):

    valid = val_mask[:, i] == 1

    y_true = val_true[valid, i]
    y_pred = val_predictions[valid, i]

    # AUC requires both classes to be present
    if len(np.unique(y_true)) < 2:
        print(f"{finding:20s} AUC: N/A")
        continue

    auc = roc_auc_score(
        y_true,
        y_pred
    )

    print(
        f"{finding:20s} "
        f"AUC: {auc:.4f} "
        f"n={len(y_true)}"
    )

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

thresholds = {}

for i, finding in enumerate(FINDINGS):
    valid = val_mask[:, i] == 1

    y_true = val_true[valid, i]
    y_pred = val_predictions[valid, i]

    precision, recall, thresh = precision_recall_curve(
        y_true,
        y_pred
    )

    # F1 for each threshold
    f1 = 2 * precision[:-1] * recall[:-1] / (
        precision[:-1] + recall[:-1] + 1e-8
    )

    best_idx = np.argmax(f1)
    best_threshold = thresh[best_idx]
    best_f1 = f1[best_idx]

    thresholds[finding] = best_threshold

    print(
        f"{finding:20s} "
        f"threshold={best_threshold:.3f} "
        f"F1={best_f1:.3f}"
    )

In [ ]:
test_predictions = cxr_model.predict(test_ds)

print("Test prediction shape:", test_predictions.shape)

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import numpy as np

test_true = test_labels
test_mask = test_masks

print("FINAL TEST RESULTS")
print("=" * 60)

for i, finding in enumerate(FINDINGS):

    valid = test_mask[:, i] == 1

    y_true = test_true[valid, i]
    y_prob = test_predictions[valid, i]

    # ROC-AUC
    if len(np.unique(y_true)) >= 2:
        auc = roc_auc_score(y_true, y_prob)
    else:
        auc = np.nan

    # Use threshold selected ONLY from validation set
    threshold = thresholds[finding]
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n{finding.upper()}")
    print(f"ROC-AUC   : {auc:.4f}")
    print(f"Threshold : {threshold:.3f}")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["Negative", "Positive"],
            digits=4,
            zero_division=0
        )
    )

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import numpy as np

print("FINAL TEST SUMMARY")
print("=" * 90)

results = []

for i, finding in enumerate(FINDINGS):

    valid = test_mask[:, i] == 1

    y_true = test_true[valid, i]
    y_prob = test_predictions[valid, i]

    threshold = thresholds[finding]
    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_prob)
    precision = precision_score(
        y_true, y_pred, zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred
    ).ravel()

    results.append({
        "Finding": finding,
        "AUC": auc,
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "N": len(y_true)
    })

results_df = pd.DataFrame(results)

print(
    results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

In [ ]:
print("LABEL DISTRIBUTION")
print("=" * 80)

for finding in FINDINGS:

    col = f"label_{finding}"

    print(f"\n{finding.upper()}")

    for name, frame in [
        ("Train", train_df),
        ("Validation", val_df),
        ("Test", test_df)
    ]:

        values = frame[col]

        positive = (values == 1).sum()
        negative = (values == 0).sum()
        uncertain = (values == -1).sum()
        missing = values.isna().sum()

        print(
            f"{name:12s} "
            f"Positive={positive:4d} "
            f"Negative={negative:4d} "
            f"Uncertain={uncertain:4d} "
            f"Missing={missing:4d}"
        )

In [ ]:
print("TRAIN:", train_df["image_path"].head(3).tolist())
print("VAL  :", val_df["image_path"].head(3).tolist())
print("TEST :", test_df["image_path"].head(3).tolist())

In [ ]:
for name, frame in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    valid_paths = frame["image_path"].apply(
        lambda p: Path(p).exists()
    )

    print(
        f"{name}: "
        f"{valid_paths.sum()}/{len(valid_paths)} "
        f"images exist"
    )

In [ ]:
def image_statistics(frame, n=500):
    sample = frame.sample(
        min(n, len(frame)),
        random_state=42
    )

    means = []
    stds = []

    for path in sample["image_path"]:
        image = tf.io.read_file(str(path))
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.cast(image, tf.float32)

        means.append(float(tf.reduce_mean(image)))
        stds.append(float(tf.math.reduce_std(image)))

    return np.mean(means), np.std(means), np.mean(stds)


for name, frame in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    mean_mean, mean_std, avg_std = image_statistics(frame)

    print(
        f"{name:12s} "
        f"mean={mean_mean:.2f} "
        f"mean_std={mean_std:.2f} "
        f"avg_pixel_std={avg_std:.2f}"
    )

In [ ]:
print("Train columns:")
print(train_df.columns.tolist())

print("\nValidation columns:")
print(val_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

In [ ]:
MODEL_PATH = MODEL_DIR / "cxr_finding_model.keras"

cxr_model.backbone_model.save(MODEL_PATH)

print("Saved:", MODEL_PATH)

In [ ]:
print(MODEL_PATH.exists())
print(MODEL_PATH)

In [ ]:
CXR_THRESHOLDS = {
    "pneumonia": 0.3647,
    "pleural_effusion": 0.4389,
    "pneumothorax": 0.6961,
    "consolidation": 0.4255,
    "pulmonary_edema": 0.5404,
}

In [ ]:
import json

THRESHOLD_PATH = MODEL_DIR / "cxr_thresholds.json"

with open(THRESHOLD_PATH, "w") as f:
    json.dump(CXR_THRESHOLDS, f, indent=4)

print("Saved:", THRESHOLD_PATH)

In [ ]:
MODEL_PATH = MODEL_DIR / "cxr_finding_model.keras"

cxr_model.backbone_model.save(MODEL_PATH)

print("Model saved:", MODEL_PATH)
print("Exists:", MODEL_PATH.exists())

In [ ]:
import json

CXR_THRESHOLDS = {
    "pneumonia": 0.3647,
    "pleural_effusion": 0.4389,
    "pneumothorax": 0.6961,
    "consolidation": 0.4255,
    "pulmonary_edema": 0.5404,
}

THRESHOLD_PATH = MODEL_DIR / "cxr_thresholds.json"

with open(THRESHOLD_PATH, "w") as f:
    json.dump(CXR_THRESHOLDS, f, indent=4)

print("Thresholds saved:", THRESHOLD_PATH)